# Record & Label — single-notebook BC data collection

Replaces the unimplemented `rover_recorder` ROS node. Runs **outside ROS**, directly against jetcam, so you can iterate fast.

**Part A — Record**: opens dual CSI cameras, rectifies the left stream using the frozen `stereo_calib.yaml`, saves JPEGs + `annotation.txt` rows while you drive with the keyboard.  
**Part B — Label**: click the road-center point on each saved image; `xpos, ypos` get filled in.

**Output format** (matches PROJECT_PLAN.md §4):  
`filename xpos ypos segment steer_tel speed_tel`

**Keyboard** (Part A):
- arrow up/down: speed ±
- arrow left/right: steer ±
- space: zero
- `1`/`2`/`3`: segment = common / left / right
- `p`: segment = pause (frames dropped)
- `r`: toggle recording
- `esc`: stop


## Part A — Record

In [1]:
# --- Config ---
from pathlib import Path
SESSION_NAME = "session01"
DATA_ROOT    = Path.home() / "rover_data"
FPS          = 15
CAM_W, CAM_H = 1280, 720
ENABLE_MOTOR = True            # True = actually drive the rover while recording
UART_PORT    = "/dev/ttyUSB0"
UART_BAUD    = 115200
CALIB_PATH   = Path.home() / "team/main/ros2_ws/src/rover_stereo/config/stereo_calib.yaml"


In [2]:
# --- Imports + path setup ---
import sys, os, time, threading
sys.path.insert(0, str(Path.home() / "team/calibration"))   # for `camera`
sys.path.insert(0, str(Path.home() / "HYU-ECL3003/rover"))  # for base_ctrl

import numpy as np
import cv2
import yaml
import ipywidgets as widgets
from IPython.display import display

from camera import Camera
from pynput import keyboard


In [3]:
# --- Load frozen calibration and build rectify maps ---
c = yaml.safe_load(open(CALIB_PATH))
K1, D1 = np.array(c["K1"]), np.array(c["D1"])
K2, D2 = np.array(c["K2"]), np.array(c["D2"])
R, T   = np.array(c["R"]),  np.array(c["T"])
W, H   = c["image_size"]

R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(K1, D1, K2, D2, (W, H), R, T, alpha=0.0)
m1x, m1y = cv2.initUndistortRectifyMap(K1, D1, R1, P1, (W, H), cv2.CV_16SC2)
m2x, m2y = cv2.initUndistortRectifyMap(K2, D2, R2, P2, (W, H), cv2.CV_16SC2)

x1, y1, w1, h1 = roi1
x2, y2, w2, h2 = roi2
xa, ya = max(x1, x2), max(y1, y2)
xb, yb = min(x1 + w1, x2 + w2), min(y1 + h1, y2 + h2)
ROI = (xa, ya, xb - xa, yb - ya) if (xb > xa and yb > ya) else (0, 0, W, H)
print("rectify ROI:", ROI, "  fx:", round(float(P1[0, 0]), 2))


rectify ROI: (0, 0, 1280, 720)   fx: 1288.22


In [4]:
# --- Open cameras ---
cam_l = Camera(sensor_id=0, capture_width=CAM_W, capture_height=CAM_H, capture_fps=FPS*2)
cam_r = Camera(sensor_id=1, capture_width=CAM_W, capture_height=CAM_H, capture_fps=FPS*2)
print("cameras up:", cam_l.running(), cam_r.running())


GST_ARGUS: Creating output stream
CONSUMER: Waiting until producer is connected...
GST_ARGUS: Available Sensor modes :
GST_ARGUS: 3280 x 2464 FR = 21.000000 fps Duration = 47619048 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 3280 x 1848 FR = 28.000001 fps Duration = 35714284 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1920 x 1080 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1640 x 1232 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1280 x 720 FR = 59.999999 fps Duration = 16666667 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: Running with following settings:
   Camera index = 0 
   Camera mode  = 4 
   Output Stream W = 1280 H = 7

[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (1100) open OpenCV | GStreamer warning: Cannot query video position: status=0, value=-1, duration=-1


GST_ARGUS: Cleaning up
CONSUMER: Done Success
GST_ARGUS: Done Success


[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (1390) setProperty OpenCV | GStreamer warning: GStreamer: unhandled property


GST_ARGUS: Creating output stream
CONSUMER: Waiting until producer is connected...
GST_ARGUS: Available Sensor modes :
GST_ARGUS: 3280 x 2464 FR = 21.000000 fps Duration = 47619048 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 3280 x 1848 FR = 28.000001 fps Duration = 35714284 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1920 x 1080 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1640 x 1232 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1280 x 720 FR = 59.999999 fps Duration = 16666667 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: Running with following settings:
   Camera index = 0 
   Camera mode  = 4 
   Output Stream W = 1280 H = 7

[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (1100) open OpenCV | GStreamer warning: Cannot query video position: status=0, value=-1, duration=-1


GST_ARGUS: Cleaning up
CONSUMER: Done Success
GST_ARGUS: Done Success
GST_ARGUS: Creating output stream
CONSUMER: Waiting until producer is connected...
GST_ARGUS: Available Sensor modes :
GST_ARGUS: 3280 x 2464 FR = 21.000000 fps Duration = 47619048 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 3280 x 1848 FR = 28.000001 fps Duration = 35714284 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1920 x 1080 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1640 x 1232 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1280 x 720 FR = 59.999999 fps Duration = 16666667 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: Running with following settings:
   

[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (1390) setProperty OpenCV | GStreamer warning: GStreamer: unhandled property


cameras up: True True


In [5]:
# --- Motor (optional) ---
motor = None
if ENABLE_MOTOR:
    from base_ctrl import BaseController
    motor = BaseController(UART_PORT, UART_BAUD)
    print("motor ready on", UART_PORT)
else:
    print("ENABLE_MOTOR=False — recording without driving")


motor ready on /dev/ttyUSB0


In [6]:
# --- Shared state + keyboard listener ---
state = dict(steer=0.0, speed=0.0, segment="common", recording=False, stop=False)

def _key_to_str(key):
    try: return key.char
    except AttributeError: return str(key).replace("Key.", "")

STEER_STEP = 0.05
SPEED_STEP = 0.05

def on_press(key):
    k = _key_to_str(key)
    if   k == "up":    state["speed"] = min( 1.0, state["speed"] + SPEED_STEP)
    elif k == "down":  state["speed"] = max(-1.0, state["speed"] - SPEED_STEP)
    elif k == "left":  state["steer"] = max(-1.0, state["steer"] - STEER_STEP)
    elif k == "right": state["steer"] = min( 1.0, state["steer"] + STEER_STEP)
    elif k == "space": state["steer"] = 0.0; state["speed"] = 0.0
    elif k == "1":     state["segment"] = "common"
    elif k == "2":     state["segment"] = "left"
    elif k == "3":     state["segment"] = "right"
    elif k == "p":     state["segment"] = "pause"
    elif k == "r":     state["recording"] = not state["recording"]
    elif k == "esc":   state["stop"] = True

listener = keyboard.Listener(on_press=on_press)
listener.start()
print("keyboard listener up — focus the desktop (not the browser tab) for OS-level keys")


keyboard listener up — focus the desktop (not the browser tab) for OS-level keys


In [7]:
# --- Recording loop ---
# Run this cell to start. Press `r` to begin saving, `esc` to stop.
session_dir = DATA_ROOT / f"{SESSION_NAME}_{time.strftime('%Y%m%d_%H%M%S')}"
(session_dir / "images").mkdir(parents=True, exist_ok=True)
ann_path = session_dir / "annotation.txt"
ann = open(ann_path, "w")
ann.write("# filename xpos ypos segment steer_tel speed_tel\n")

preview = widgets.Image(format="jpeg", width=640)
status  = widgets.HTML()
display(widgets.VBox([preview, status]))

dt = 1.0 / FPS
saved = 0
state["stop"] = False
try:
    while not state["stop"]:
        t0 = time.time()
        L = cam_l.read()
        R_ = cam_r.read()
        if L is None or R_ is None:
            time.sleep(dt); continue

        # rectify left (the only stream we save)
        lr = cv2.remap(L, m1x, m1y, cv2.INTER_LINEAR)
        rx, ry, rw, rh = ROI
        lr_roi = lr[ry:ry + rh, rx:rx + rw]

        if state["recording"] and state["segment"] != "pause":
            ts = time.time_ns() // 1_000_000
            fname = f"{ts}.jpg"
            cv2.imwrite(
                str(session_dir / "images" / fname),
                cv2.cvtColor(lr_roi, cv2.COLOR_RGB2BGR),
                [cv2.IMWRITE_JPEG_QUALITY, 90],
            )
            # xpos/ypos = -1 placeholder, filled in Part B
            ann.write(f"{fname} -1 -1 {state['segment']} "
                      f"{state['steer']:.3f} {state['speed']:.3f}\n")
            ann.flush()
            saved += 1

        if hasattr(motor, "base_speed_ctrl"):
            # Firmware range: -0.5..+0.5 (0.5 = 100% PWM). Positive = forward,
            # per HYU-ECL3003/rover/tutorial_ctrl.ipynb cell 5.
            thr = state["speed"]
            L_m = thr * (1 - state["steer"])
            R_m = thr * (1 + state["steer"])
            L_m = max(-0.5, min(0.5, L_m))
            R_m = max(-0.5, min(0.5, R_m))
            motor.base_speed_ctrl(L_m, R_m)

        small = cv2.resize(lr_roi, (640, int(640 * lr_roi.shape[0] / lr_roi.shape[1])))
        ok, buf = cv2.imencode(".jpg", cv2.cvtColor(small, cv2.COLOR_RGB2BGR))
        if ok: preview.value = buf.tobytes()
        rec = "<b style='color:red'>REC</b>" if state["recording"] else "<b>IDLE</b>"
        status.value = (f"{rec} &nbsp; saved={saved} &nbsp; "
                        f"segment=<b>{state['segment']}</b> &nbsp; "
                        f"steer={state['steer']:+.2f} &nbsp; speed={state['speed']:+.2f}")

        slack = dt - (time.time() - t0)
        if slack > 0: time.sleep(slack)
finally:
    if hasattr(motor, "base_speed_ctrl"): motor.base_speed_ctrl(0, 0)
    ann.close()
    print(f"\nsession: {session_dir}\nframes saved: {saved}")



session: /home/ircv16/rover_data/session01_20260521_164830
frames saved: 0


KeyboardInterrupt: 

In [1]:
# --- Cleanup (run when done) ---
listener.stop()
cam_l.stop(); cam_r.stop()
print("cameras + listener released")


NameError: name 'listener' is not defined

## Part B — Label

Click the road-center point on each image. `xpos, ypos` get written back to `annotation.txt`.  
Set `SESSION_DIR` to the path printed by Part A (or leave as `session_dir` to label what you just recorded).


In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt

SESSION_DIR = session_dir   # or: Path.home() / "rover_data/session01_YYYYMMDD_HHMMSS"
ANN = SESSION_DIR / "annotation.txt"

# Load annotation rows (preserve comment header)
header_lines, rows = [], []
for line in open(ANN):
    s = line.strip()
    if not s or s.startswith("#"):
        header_lines.append(line.rstrip("\n"))
    else:
        rows.append(s.split())
print(f"{len(rows)} rows to label")


In [ ]:
idx = [0]
fig, ax = plt.subplots(figsize=(9, 5))

def show(i):
    ax.clear()
    if i >= len(rows):
        ax.set_title("DONE — run the next cell to save")
        fig.canvas.draw(); return
    fname = rows[i][0]
    img = cv2.imread(str(SESSION_DIR / "images" / fname))
    if img is None:
        ax.set_title(f"MISSING {fname} — skipping")
        idx[0] += 1; show(idx[0]); return
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    xp, yp = rows[i][1], rows[i][2]
    if xp != "-1":
        ax.plot(int(xp), int(yp), "r+", markersize=20, mew=2)
    ax.set_title(f"{i + 1}/{len(rows)}  {fname}  seg={rows[i][3]}  "
                 f"steer={rows[i][4]}  click road center")
    ax.set_xticks([]); ax.set_yticks([])
    fig.canvas.draw()

def onclick(event):
    if event.inaxes != ax or event.xdata is None: return
    i = idx[0]
    if i >= len(rows): return
    rows[i][1] = str(int(event.xdata))
    rows[i][2] = str(int(event.ydata))
    idx[0] += 1
    show(idx[0])

def onkey(event):
    # back: undo last click; skip: leave as -1 and advance
    if event.key == "b" and idx[0] > 0:
        idx[0] -= 1
        rows[idx[0]][1] = "-1"; rows[idx[0]][2] = "-1"
    elif event.key == "n":
        idx[0] += 1
    show(idx[0])

fig.canvas.mpl_connect("button_press_event", onclick)
fig.canvas.mpl_connect("key_press_event",    onkey)
show(0)
print("click: label  |  n: skip  |  b: back+unlabel")


In [ ]:
# --- Save labeled annotation ---
with open(ANN, "w") as f:
    for h in header_lines: f.write(h + "\n")
    for r in rows:         f.write(" ".join(r) + "\n")
labeled = sum(1 for r in rows if r[1] != "-1")
print(f"saved: {labeled} / {len(rows)} frames labeled in {ANN}")
